## Собираем сегменты

In [4]:
import pandas as pd
import numpy as np
from itertools import combinations
from datetime import datetime

In [8]:
current_year = datetime.now().year
df_merged = pd.read_csv('data/processed/processed_task1_total_liquidity.csv')
# ==========================================
# 1. БИННИНГ ЧИСЛОВЫХ ПРИЗНАКОВ (БЕЗ delta_p)
# ==========================================
bins_price = [0, 600_000, 1_500_000, 3_000_000, 6_000_000, 10_000_000, np.inf]
labels_price = ['Эконом (<600к)', 'Комфорт (600к-1.5м)', 'Комфорт+ (1.5м-3м)', 'Бизнес (3м-6м)', 'Премиум (6м-10м)', 'Люкс (>10м)']
df_merged['price_bin'] = pd.cut(df_merged['price'], bins=bins_price, labels=labels_price)

bins_mileage = [-1, 50_000, 100_000, 150_000, 250_000, 521_231]
labels_mileage = ['Свежое авто (0-50к)', 'Оптимальный б/у (50-100к)', 'Психологический барьер (100-150к)', 'Рабочая лошадка (150-250к)', 'Глубокое б/у (250-520 231к)']
df_merged['mileage_bin'] = pd.cut(df_merged['mileage'], bins=bins_mileage, labels=labels_mileage)

bins_year = [1900, current_year - 15, current_year - 7, current_year - 3, current_year + 1]
labels_year = ['Старше 15 лет (<2010)', 'Бюджет 7-15 лет (2010-2017)', 'Масс-маркет 3-7 лет (2018-2021)', 'Свежие <3 лет (2022-2025)']
df_merged['year_bin'] = pd.cut(df_merged['year'], bins=bins_year, labels=labels_year)


# ==========================================
# 2. ФОРМИРОВАНИЕ КОМБИНАЦИЙ
# ==========================================
features = [
    'price_bin', 'year_bin', 'mileage_bin', 
    'region', 'transmission', 'brand', 'body_type', 'drive', 'fuel_type', 'condition'
]

all_combos = list(combinations(features, 4))

# Бизнес-логика: обязательно регион И (цена ИЛИ бренд)
valid_combos = []
for combo in all_combos:
    if 'region' in combo and ('price_bin' in combo or 'brand' in combo):
        valid_combos.append(list(combo))

print(f"Всего возможных комбинаций: {len(all_combos)}")
print(f"Валидных комбинаций для проверки: {len(valid_combos)}\n")

# ==========================================
# 3. ПОИСК И АГРЕГАЦИЯ
# ==========================================
all_segments_results = []

for combo in valid_combos:
    grouped = df_merged.groupby(combo, observed=True).agg(
        listings_count=('id', 'count'),
        liquidity=('total_contacts', 'median')
    ).reset_index()
    
    # Фильтруем мусор (строго >= 100)
    valid_segments = grouped[grouped['listings_count'] >= 100].copy()
    
    if not valid_segments.empty:
        # Собираем портрет сегмента в одну строку
        valid_segments['segment_portrait'] = valid_segments[combo].astype(str).agg(' | '.join, axis=1)
        
        # Оставляем только нужные колонки
        res = valid_segments[['segment_portrait', 'listings_count', 'liquidity']].copy()
        res['used_features'] = ' + '.join(combo)
        
        all_segments_results.append(res)

# ==========================================
# 4. ФИНАЛЬНАЯ СБОРКА И ВЫГРУЗКА
# ==========================================
if all_segments_results:
    final_report = pd.concat(all_segments_results, ignore_index=True)
    
    # Чистим дубли
    final_report = final_report.drop_duplicates(subset=['segment_portrait'])
    
    # Сортируем по ликвидности по убыванию
    final_report = final_report.sort_values(by='liquidity', ascending=False).reset_index(drop=True)
    
    # Оставляем liquidity как float, но округляем до 1 знака
    final_report['liquidity'] = final_report['liquidity'].round(1)
    
    # Кол-во объявлений жестко делаем int
    final_report['listings_count'] = final_report['listings_count'].astype(int)
    
    # === СОХРАНЯЕМ ВООБЩЕ ВСЁ В ФАЙЛ ===
    final_report.to_csv('data/processed/all_segments_report.csv', index=False)
    
    n_total = len(final_report)
    print(f"Успешно. Всего найдено статистически плотных сегментов: {n_total}")
    print("ВЕСЬ список сохранен в 'data/processed/all_segments_report.csv' (передай Мишгану)\n")
    
    # === ВЫВОДИМ ТОЛЬКО 30 ДЛЯ ЭКРАНА ===
    cols_to_show = ['segment_portrait', 'liquidity', 'listings_count', 'used_features']
    
    top_10 = final_report[cols_to_show].head(10)
    
    mid_idx = n_total // 2
    mid_10 = final_report[cols_to_show].iloc[mid_idx - 5 : mid_idx + 5]
    
    bottom_10 = final_report[cols_to_show].tail(10)
    
    print("Топ-10 САМЫХ ЛИКВИДНЫХ сегментов (Высокий спрос):")
    display(top_10)
    
    print("\n10 СРЕДНИХ сегментов (Норма рынка / Базлайн):")
    display(mid_10)
    
    print("\nТоп-10 САМЫХ ХУДШИХ сегментов (Кандидаты на ранний VAS):")
    display(bottom_10)

else:
    print("Ни один сегмент не прошел фильтр в 100 объявлений.")

Всего возможных комбинаций: 210
Валидных комбинаций для проверки: 49

Успешно. Всего найдено статистически плотных сегментов: 33806
ВЕСЬ список сохранен в 'data/processed/all_segments_report.csv' (передай Мишгану)

Топ-10 САМЫХ ЛИКВИДНЫХ сегментов (Высокий спрос):


,segment_portrait,liquidity,listings_count,used_features
0,Эконом (<600к) | Московская область | Mercedes...,27.0,103,price_bin + region + brand + body_type
1,Комфорт (600к-1.5м) | Дагестан | Mercedes-Benz...,27.0,135,price_bin + region + brand + drive
2,Комфорт (600к-1.5м) | Старше 15 лет (<2010) | ...,25.0,167,price_bin + year_bin + region + brand
3,Эконом (<600к) | Москва | Фургон | Задний,25.0,143,price_bin + region + body_type + drive
4,Эконом (<600к) | Московская область | Mercedes...,25.0,124,price_bin + region + brand + drive
5,Комфорт (600к-1.5м) | Дагестан | Автомат | Mer...,24.0,161,price_bin + region + transmission + brand
6,Эконом (<600к) | Московская область | Mercedes...,24.0,159,price_bin + region + brand + condition
7,Эконом (<600к) | Старше 15 лет (<2010) | Моско...,24.0,171,price_bin + year_bin + region + brand
8,Комфорт (600к-1.5м) | Дагестан | Mercedes-Benz...,24.0,155,price_bin + region + brand + condition
9,Эконом (<600к) | Московская область | Mercedes...,24.0,129,price_bin + region + brand + fuel_type



10 СРЕДНИХ сегментов (Норма рынка / Базлайн):


,segment_portrait,liquidity,listings_count,used_features
16898,Рабочая лошадка (150-250к) | Пермский край | F...,7.0,132,mileage_bin + region + brand + condition
16899,Рабочая лошадка (150-250к) | Пермский край | T...,7.0,115,mileage_bin + region + brand + condition
16900,Рабочая лошадка (150-250к) | Ростовская област...,7.0,219,mileage_bin + region + brand + condition
16901,Москва | Mazda | Передний | Бензин,7.0,603,region + brand + drive + fuel_type
16902,Москва | Nissan | Полный | Дизель,7.0,110,region + brand + drive + fuel_type
16903,Москва | УАЗ | Полный | Бензин,7.0,100,region + brand + drive + fuel_type
16904,Оптимальный б/у (50-100к) | Ставропольский кра...,7.0,116,mileage_bin + region + brand + drive
16905,Оптимальный б/у (50-100к) | Тамбовская область...,7.0,158,mileage_bin + region + brand + drive
16906,Эконом (<600к) | Старше 15 лет (<2010) | Алтай...,7.0,114,price_bin + year_bin + region + condition
16907,Эконом (<600к) | Старше 15 лет (<2010) | Алтай...,7.0,1530,price_bin + year_bin + region + condition



Топ-10 САМЫХ ХУДШИХ сегментов (Кандидаты на ранний VAS):


,segment_portrait,liquidity,listings_count,used_features
33796,Свежое авто (0-50к) | Приморский край | Toyota...,0.0,774,mileage_bin + region + brand + fuel_type
33797,Свежое авто (0-50к) | Приморский край | Nissan...,0.0,185,mileage_bin + region + brand + fuel_type
33798,Свежое авто (0-50к) | Приморский край | Merced...,0.0,255,mileage_bin + region + brand + fuel_type
33799,Масс-маркет 3-7 лет (2018-2021) | Приморский к...,0.0,205,year_bin + region + brand + drive
33800,Масс-маркет 3-7 лет (2018-2021) | Приморский к...,0.0,123,year_bin + region + brand + drive
33801,Масс-маркет 3-7 лет (2018-2021) | Приморский к...,0.0,151,year_bin + region + brand + drive
33802,Комфорт+ (1.5м-3м) | Рабочая лошадка (150-250к...,0.0,109,price_bin + mileage_bin + region + drive
33803,Комфорт+ (1.5м-3м) | Рабочая лошадка (150-250к...,0.0,133,price_bin + mileage_bin + region + drive
33804,Бизнес (3м-6м) | Приморский край | Седан | Полный,0.0,116,price_bin + region + body_type + drive
33805,Масс-маркет 3-7 лет (2018-2021) | Приморский к...,0.0,245,year_bin + region + brand + drive
